# Gaussian Entropic Interpolation

For Gaussian marginals, balanced entropic OT stays inside a finite-dimensional Gaussian family.  With quadratic cost, the optimal entropic coupling has cross-covariance
$$
K_\varepsilon=A^{1/2}U\operatorname{diag}(s_i)V^\top B^{1/2},\qquad
s_i=\frac{\sqrt{\varepsilon^2+16\sigma_i^2}-\varepsilon}{4\sigma_i},
$$
where $A^{1/2}B^{1/2}=U\operatorname{diag}(\sigma_i)V^\top$.  The figure compares the deterministic Bures interpolation with the broader stochastic interpolation obtained from this entropic Gaussian coupling.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    DIRAC_MARKER_SIZE,
    GRAY,
    LIGHT_GRAY,
    ORANGE,
    RED,
    VIOLET,
    box_axes,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

from matplotlib.patches import Ellipse

NAME = "sinkhorn-gaussian-regularized-geodesic"
out = figure_dir(NAME)


We use two anisotropic two-dimensional covariance matrices.  The unregularized panel uses the Bures transport map.  The entropic panel uses the covariance of $(1-t)X+tY$ under the optimal entropic Gaussian coupling.


In [ ]:
def rot(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

def spd_sqrt(A):
    w, V = np.linalg.eigh(A)
    return (V * np.sqrt(np.maximum(w, 0))) @ V.T

def spd_invsqrt(A):
    w, V = np.linalg.eigh(A)
    return (V * (1 / np.sqrt(np.maximum(w, 1e-15)))) @ V.T

def ellipse_params(Sigma, nsig=2.0):
    w, V = np.linalg.eigh(Sigma)
    order = np.argsort(w)[::-1]
    w, V = w[order], V[:, order]
    angle = np.degrees(np.arctan2(V[1, 0], V[0, 0]))
    return 2*nsig*np.sqrt(w[0]), 2*nsig*np.sqrt(w[1]), angle

m0 = np.array([-1.25, -0.18])
m1 = np.array([1.15, 0.32])
A = rot(np.deg2rad(25)) @ np.diag([0.25, 0.045]) @ rot(np.deg2rad(25)).T
B = rot(np.deg2rad(-38)) @ np.diag([0.055, 0.24]) @ rot(np.deg2rad(-38)).T

def bures_covariances(ts):
    Ah = spd_sqrt(A)
    T = spd_invsqrt(A) @ spd_sqrt(Ah @ B @ Ah) @ spd_invsqrt(A)
    covs = []
    for t in ts:
        M = (1-t)*np.eye(2) + t*T
        covs.append(M @ A @ M.T)
    return covs

def entropic_covariances(ts, eps=0.42):
    Ah, Bh = spd_sqrt(A), spd_sqrt(B)
    U, sig, Vt = np.linalg.svd(Ah @ Bh)
    scoef = (np.sqrt(eps**2 + 16*sig**2) - eps) / (4*sig)
    K = Ah @ U @ np.diag(scoef) @ Vt @ Bh
    covs = []
    for t in ts:
        covs.append((1-t)**2*A + t**2*B + t*(1-t)*(K + K.T))
    return covs


The ellipses are drawn at the interpolated means and colored from red to blue as $t$ goes from $0$ to $1$.


In [ ]:
ts = np.linspace(0, 1, 5)

def draw_ellipses(covs, filename):
    fig, ax = plt.subplots(figsize=(2.65, 2.05))
    for t, Sigma in zip(ts, covs):
        m = (1-t)*m0 + t*m1
        width, height, angle = ellipse_params(Sigma, nsig=1.7)
        e = Ellipse(m, width, height, angle=angle, fill=False, lw=1.15, edgecolor=interp_color(float(t)))
        ax.add_patch(e)
        ax.scatter([m[0]], [m[1]], s=DIRAC_MARKER_SIZE*0.72, color=interp_color(float(t)), marker="o", edgecolor="none", linewidth=0, zorder=3)
    ax.plot([m0[0], m1[0]], [m0[1], m1[1]], color=LIGHT_GRAY, lw=0.9, zorder=0)
    ax.set_xlim(-2.28, 2.28)
    ax.set_ylim(-1.18, 1.32)
    ax.set_aspect("equal")
    remove_axes(ax)
    save_pdf(fig, out / filename, pad_inches=0.055)
    plt.close(fig)

draw_ellipses(bures_covariances(ts), "bures.pdf")
draw_ellipses(entropic_covariances(ts), "entropic.pdf")
